#Customers Data Ingestion

##Requirments
###1. Streaming data( continous data)
###2. DLT table
###3. Using Auto-loader (incremental data load)
###4. Adding metadata columns - (file_path and insert_timestamp)
###5. Language - SQL
###6. Ingest all records
###7. Implement DQ(Data Qulaity) Rules

In [0]:
CREATE OR REFRESH STREAMING TABLE bronze_customers
COMMENT 'Bronze table for customers'
TBLPROPERTIES ('quality'='bronze')
AS 
 SELECT * ,
 _metadata.file_path AS file_path,
 current_timestamp() AS ingestion_timestamp
 FROM 
 cloud_files('/Volumes/circuitbox/landing/operational_data/customer/',
             'json',
            map('cloudFiles.inferColumnTypes','true')
            );


Now data needs to be loaded to Silver layer

requirements-
1. Transformation
  - CAST date_of_birth to Date
  - CAST created_date to Date

2. Data Qulaity(DQ) Rules using Expectation

    a. Fail if customer_id is null

    b. Drop rexords with cutsomer_name is null

    c. Warn if telephone is less than 10 characters

    d. Warn if email is Null
    
    e. Warn if date_of_birth is before 1920

3. Only bring latest records using STREAM modifer keyword at source table.

4. LIVE keyword - that denotes same schema in same ETL Pipeline
    

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_customers_clean (
  CONSTRAINT Valid_customer_id EXPECT (customer_id is not null) ON VIOLATION FAIL UPDATE,
  CONSTRAINT Valid_customer_name EXPECT (customer_name is NOT null) ON VIOLATION DROP ROW,
  CONSTRAINT Valid_phone_no EXPECT (length(telephone) >= 10 ),
  CONSTRAINT Valid_email_id EXPECT (email is not null ),
  CONSTRAINT Valid_date_of_birth EXPECT (date_of_birth >= '1920-01-01' )
)
COMMENT 'Silver clean table for customers and Applying DQ rules'
TBLPROPERTIES ('quality'='silver clean')
AS
Select 
  customer_id,
  customer_name,
  date_of_birth:: DATE,
  telephone,
  email,
  created_date :: DATE
from STREAM(LIVE.bronze_customers)

###Notes
1. Apply SCD(Slowly Changin Dimensions) Type 1 means updating the existing records. using APPLY Changes Into API.
2. Apply Changes Into doesn't cretae table, therefore first nee dto create a empty table.
3. SCD Tyoe 1 is the deafult value if you don't mention it.
4. Currenylt DLT handles only SCDP type 1 and 2 only.

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_customers
COMMENT ' SCD Type 1 Silver table for customers'
TBLPROPERTIES ('quality'='silver')


In [0]:
APPLY CHANGES INTO LIVE.silver_customers
FROM STREAM(LIVE.silver_customers_clean)
KEYS(customer_id)
SEQUENCE BY created_date
STORED AS SCD TYPE 1; --Opption as Default value is SCD TYPE 1
